# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset is provided by a Croissant schema at the following URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View dataset metadata
meta = dataset.metadata
print(f"Name: {meta.name}\n\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values from the dataset.

In [ ]:
# List all available record sets and their @id
print("Available record sets (by @id):")
record_sets = dataset.list_record_sets()
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name','')} (type: {rs.get('@type','')})")
    # Print the fields available in this record set
    fields = dataset.list_fields(record_set=rs['@id'])
    print("    Fields (@id) and names:")
    for field in fields:
        print(f"      • {field['@id']} | {field.get('name','')} | type: {field.get('dataType','')}")
    print()

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Make sure to reference record set and field entities by their `@id`.

In [ ]:
# For demonstration, enumerate all record sets and load each as a DataFrame.
record_set_ids = [rs['@id'] for rs in dataset.list_record_sets()]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns in the primary/first record set (usually main tabular data)
if record_set_ids:
    print(f"Columns in record set '{record_set_ids[0]}':")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply exploratory data analysis steps such as filtering, normalization, and grouping on the main record set.

Identify a numeric field by its `@id` (from above cell's printed fields). For demonstration, we'll attempt to automatically select a numeric field (float or integer), otherwise, please customize `numeric_field_id` below using the printed field info.

In [ ]:
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Try to heuristically pick a numeric field
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fallback or prompt for manual assignment
    numeric_field_id = df.select_dtypes('number').columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Set a threshold (example: 10) for filtering
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records in '{main_record_set_id}' where {numeric_field_id} > {threshold} (showing top 5):")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"First 5 values for normalized {numeric_field_id}:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another field (e.g., a categorical field)
categorical_col = None
for col in df.columns:
    if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
        categorical_col = col
        break
group_field = categorical_col
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    print(f"Grouped mean of '{numeric_field_id}' by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the normalized numeric field and the group means (if grouping was successful).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Histogram of normalized values
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=30, kde=True)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Count")
plt.show()

# If grouped_df exists, visualize group means
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(12,5))
    grouped_df.sort_values(numeric_field_id, ascending=False).head(20).plot(
        kind='bar', legend=False)
    plt.title(f"Mean {numeric_field_id} by {group_field} (top 20)")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, you:
- Loaded FAIR² protein mass spectrometry dataset metadata and tables via `mlcroissant`
- Explored available record sets, their fields, and referenced all entities by their `@id`
- Loaded record data into DataFrames and identified numeric and categorical fields
- Applied basic filtering, normalization, and groupwise analysis
- Visualized data distributions and group means

This approach can be extended to more advanced analyses as needed for protein abundance and modification research!